# संभाव्यता और सांख्यिकी का परिचय
इस नोटबुक में, हम कुछ ऐसे अवधारणाओं के साथ खेलेंगे जिन पर हमने पहले चर्चा की है। संभाव्यता और सांख्यिकी की कई अवधारणाएं पायथन में डेटा प्रोसेसिंग के प्रमुख लाइब्रेरीज जैसे `numpy` और `pandas` में अच्छी तरह से मौजूद हैं।


In [ ]:
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt

## यादृच्छिक चर और वितरण
चलिए 0 से 9 तक के समान वितरण से 30 मानों का एक नमूना निकालना शुरू करते हैं। हम माध्य और वैरिएंस भी गणना करेंगे।


In [ ]:
sample = [ random.randint(0,10) for _ in range(30) ]
print(f"Sample: {sample}")
print(f"Mean = {np.mean(sample)}")
print(f"Variance = {np.var(sample)}")

नमूने में कितने अलग-अलग मान हैं, इसका दृश्यात्मक अनुमान लगाने के लिए, हम **हिस्टोग्राम** बना सकते हैं:


In [ ]:
plt.hist(sample)
plt.show()

## वास्तविक डेटा का विश्लेषण

वास्तविक दुनिया के डेटा का विश्लेषण करते समय माध्य और विचरण बहुत महत्वपूर्ण होते हैं। आइए बेसबॉल खिलाड़ियों के बारे में डेटा [SOCR MLB Height/Weight Data](http://wiki.stat.ucla.edu/socr/index.php/SOCR_Data_MLB_HeightsWeights) से लोड करें


In [ ]:
df = pd.read_csv("../../data/SOCR_MLB.tsv",sep='\t', header=None, names=['Name','Team','Role','Weight','Height','Age'])
df


> हम यहां डेटा विश्लेषण के लिए [**Pandas**](https://pandas.pydata.org/) नामक एक पैकेज का उपयोग कर रहे हैं। इस पाठ्यक्रम में हम बाद में Pandas और पायथन में डेटा के साथ काम करने के बारे में और बात करेंगे।

चलिए आयु, ऊंचाई और वजन के औसत मान की गणना करते हैं:


In [ ]:
df[['Age','Height','Weight']].mean()

अब ऊंचाई पर ध्यान केंद्रित करते हैं, और मानक विचलन तथा विचरण की गणना करते हैं: 


In [ ]:
print(list(df['Height'])[:20])

In [ ]:
mean = df['Height'].mean()
var = df['Height'].var()
std = df['Height'].std()
print(f"Mean = {mean}\nVariance = {var}\nStandard Deviation = {std}")

औसत के अलावा, माध्यिका मान और चतुर्थांशों को देखना भी समझदारी होगी। इन्हें एक **बॉक्स प्लॉट** का उपयोग करके दर्शाया जा सकता है:


In [ ]:
plt.figure(figsize=(10,2))
plt.boxplot(df['Height'].ffill(), orientation='horizontal', showmeans=True)
plt.grid(color='gray', linestyle='dotted')
plt.tight_layout()
plt.show()

हम अपने डेटासेट के उपसमूहों के भी बॉक्स प्लॉट बना सकते हैं, उदाहरण के लिए, खिलाड़ी भूमिका द्वारा समूहित। 


In [ ]:
df.boxplot(column='Height', by='Role', figsize=(10,8))
plt.xticks(rotation='vertical')
plt.tight_layout()
plt.show()

> **नोट**: यह आरेख सुझाव देता है कि औसतन, पहले बेसमैन की ऊंचाई दूसरे बेसमैन की ऊंचाई से अधिक होती है। बाद में हम सीखेंगे कि हम इस परिकल्पना का अधिक औपचारिक रूप से परीक्षण कैसे कर सकते हैं, और यह कैसे प्रदर्शित कर सकते हैं कि हमारे डेटा में यह सांख्यिकीय रूप से महत्वपूर्ण है।  

उम्र, ऊंचाई और वजन सभी सतत यादृच्छिक चर हैं। आपको क्या लगता है कि उनका वितरण क्या होगा? पता लगाने का एक अच्छा तरीका है कि मानों का हिस्टोग्राम बनाएं: 


In [ ]:
df['Weight'].hist(bins=15, figsize=(10,6))
plt.suptitle('Weight distribution of MLB Players')
plt.xlabel('Weight')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## सामान्य वितरण

आइए एक कृत्रिम वजन का नमूना बनाएं जो हमारे वास्तविक डेटा के समान माध्य और विचरण के साथ सामान्य वितरण का पालन करता है:


In [ ]:
generated = np.random.normal(mean, std, 1000)
generated[:20]

In [ ]:
plt.figure(figsize=(10,6))
plt.hist(generated, bins=15)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
plt.hist(np.random.normal(0,1,50000), bins=300)
plt.tight_layout()
plt.show()

चूँकि असल जीवन में अधिकांश मान सामान्य रूप से वितरित होते हैं, हमें नमूना डेटा उत्पन्न करने के लिए एक समान यादृच्छिक संख्या जनरेटर का उपयोग नहीं करना चाहिए। यह है जो होता है अगर हम समान वितरण के साथ वज़न उत्पन्न करने की कोशिश करें (जो `np.random.rand` द्वारा उत्पन्न होता है):  


In [ ]:
wrong_sample = np.random.rand(1000)*2*std+mean-std
plt.figure(figsize=(10,6))
plt.hist(wrong_sample)
plt.tight_layout()
plt.show()

## विश्वास अंतराल

आइए अब बेसबॉल खिलाड़ियों के वजन और ऊंचाई के लिए विश्वास अंतराल की गणना करें। हम इस कोड का उपयोग करेंगे [इस स्टैकओवरफ्लो चर्चा से](https://stackoverflow.com/questions/15033511/compute-a-confidence-interval-from-sample-data):


In [ ]:
import scipy.stats

def mean_confidence_interval(data, confidence=0.95):
    a = 1.0 * np.array(data)
    n = len(a)
    m, se = np.mean(a), scipy.stats.sem(a)
    h = se * scipy.stats.t.ppf((1 + confidence) / 2., n-1)
    return m, h

for p in [0.85, 0.9, 0.95]:
    m, h = mean_confidence_interval(df['Weight'].ffill(),p)
    print(f"p={p:.2f}, mean = {m:.2f} ± {h:.2f}")

## परिकल्पना परीक्षण

चलिए अपने बेसबॉल खिलाड़ियों के डेटासेट में विभिन्न भूमिकाओं का अन्वेषण करते हैं:


In [ ]:
df.groupby('Role').agg({ 'Weight' : 'mean', 'Height' : 'mean', 'Age' : 'count'}).rename(columns={ 'Age' : 'Count'})

आइए यह परिकल्पना परीक्षण करें कि पहला बेसमैन दूसरे बेसमैन से लंबा होता है। इसे परीक्षण करने का सबसे सरल तरीका विश्वास अंतराल का परीक्षण करना है:


In [ ]:
for p in [0.85,0.9,0.95]:
    m1, h1 = mean_confidence_interval(df.loc[df['Role']=='First_Baseman',['Height']],p)
    m2, h2 = mean_confidence_interval(df.loc[df['Role']=='Second_Baseman',['Height']],p)
    print(f'Conf={p:.2f}, 1st basemen height: {m1-h1[0]:.2f}..{m1+h1[0]:.2f}, 2nd basemen height: {m2-h2[0]:.2f}..{m2+h2[0]:.2f}')

हम देख सकते हैं कि अंतराल ओवरलैप नहीं करते हैं।

परिकल्पना को साबित करने का एक सांख्यिकीय रूप से अधिक सही तरीका है **स्टूडेंट t-परीक्षण** का उपयोग करना:


In [ ]:
from scipy.stats import ttest_ind

tval, pval = ttest_ind(df.loc[df['Role']=='First_Baseman',['Height']], df.loc[df['Role']=='Second_Baseman',['Height']],equal_var=False)
print(f"T-value = {tval[0]:.2f}\nP-value: {pval[0]}")

`ttest_ind` फ़ंक्शन द्वारा लौटाए गए दो मान हैं:
* p-मूल्य को दो वितरणों के समान माध्य होने की संभावना के रूप में माना जा सकता है। हमारे मामले में, यह बहुत कम है, जिसका अर्थ है कि पहले बेसमेन के लंबे होने के पक्ष में मजबूत साक्ष्य हैं।
* t-मूल्य सामान्यीकृत माध्य भेद का मध्यवर्ती मान है जिसका उपयोग t-टेस्ट में किया जाता है, और इसे दिए गए विश्वसनीयता मान के लिए एक सीमा मान के खिलाफ तुलना की जाती है।


## केंद्रीय सीमा प्रमेय के साथ सामान्य वितरण का सिमुलेशन करना

पाइथन में छद्म-यादृच्छिक जनरेटर हमें समान वितरण देने के लिए डिज़ाइन किया गया है। यदि हम सामान्य वितरण के लिए एक जनरेटर बनाना चाहते हैं, तो हम केंद्रीय सीमा प्रमेय का उपयोग कर सकते हैं। एक सामान्य रूप से वितरित मान प्राप्त करने के लिए हम बस समान-जनरेट किए गए नमूने का माध्य निकालेंगे।


In [ ]:
def normal_random(sample_size=100):
    sample = [random.uniform(0,1) for _ in range(sample_size) ]
    return sum(sample)/sample_size

sample = [normal_random() for _ in range(100)]
plt.figure(figsize=(10,6))
plt.hist(sample)
plt.tight_layout()
plt.show()

## सहसंबंध और evil बेसबॉल कॉर्पोरेशन

सहसंबंध हमें डेटा अनुक्रमों के बीच संबंध खोजने की अनुमति देता है। हमारे खिलौने के उदाहरण में, चलो कल्पना करते हैं कि एक evil बेसबॉल कॉर्पोरेशन है जो अपने खिलाड़ियों को उनकी ऊंचाई के अनुसार भुगतान करता है - जितना खिलाड़ी ऊँचा होता है, उतना अधिक पैसा वह/वह प्राप्त करता है। मान लीजिए कि एक आधार वेतन $1000 है, और ऊंचाई के आधार पर $0 से $100 तक का अतिरिक्त बोनस है। हम MLB के असली खिलाड़ियों को लेंगे, और उनके काल्पनिक वेतन की गणना करेंगे:


In [ ]:
heights = df['Height'].ffill()
salaries = 1000+(heights-heights.min())/(heights.max()-heights.mean())*100
print(list(zip(heights, salaries))[:10])

आइए अब उन अनुक्रमों का सहवरण (covariance) और सहसंबंध (correlation) गणना करें। `np.cov` हमें एक तथाकथित **covariance matrix** देगा, जो सहवरण का बहु-वेरिएबल्स के लिए विस्तार है। सहवरण मैट्रिक्स $M$ का तत्व $M_{ij}$ इनपुट वेरिएबल्स $X_i$ और $X_j$ के बीच सहसंबंध है, और विकर्ण मान $M_{ii}$ $X_{i}$ का प्रसरण (variance) है। इसी प्रकार, `np.corrcoef` हमें **correlation matrix** देगा।


In [ ]:
print(f"Covariance matrix:\n{np.cov(heights, salaries)}")
print(f"Covariance = {np.cov(heights, salaries)[0,1]}")
print(f"Correlation = {np.corrcoef(heights, salaries)[0,1]}")

1 के बराबर सहसंबंध का अर्थ है कि दो चरों के बीच एक मजबूत **रेखीय संबंध** है। हम एक मान को दूसरे के खिलाफ प्लॉट करके रेखीय संबंध को दृश्य रूप से देख सकते हैं:


In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(heights,salaries)
plt.tight_layout()
plt.show()

चलिए देखते हैं अगर संबंध रैखिक न हो तो क्या होगा। मान लीजिए कि हमारी कंपनी ने ऊंचाई और वेतन के बीच स्पष्ट रैखिक निर्भरता को छिपाने का फैसला किया, और सूत्र में कुछ गैर-रैखिकता, जैसे `sin`, जोड़ी: 


In [ ]:
salaries = 1000+np.sin((heights-heights.min())/(heights.max()-heights.mean()))*100
print(f"Correlation = {np.corrcoef(heights, salaries)[0,1]}")

इस मामले में, सहसंबंध थोड़ा कम है, लेकिन यह अभी भी काफी उच्च है। अब, संबंध को और भी कम स्पष्ट बनाने के लिए, हम वेतन में कुछ यादृच्छिक चर जोड़कर अतिरिक्त यादृच्छिकता जोड़ना चाह सकते हैं। देखते हैं क्या होता है:


In [ ]:
salaries = 1000+np.sin((heights-heights.min())/(heights.max()-heights.mean()))*100+np.random.random(size=len(heights))*20-10
print(f"Correlation = {np.corrcoef(heights, salaries)[0,1]}")

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(heights, salaries)
plt.tight_layout()
plt.show()

> क्या आप यह अनुमान लगा सकते हैं कि बिंदु इस तरह ऊर्ध्वाधर रेखाओं में क्यों व्यवस्थित हो जाते हैं?

हमने वेतन जैसे कृत्रिम रूप से निर्मित अवधारणा और देखे गए चर *ऊंचाई* के बीच सहसंबंध का अवलोकन किया है। आइए देखें कि क्या दो देखे गए चर, जैसे ऊंचाई और वजन, भी सहसंबंधित हैं:


In [ ]:
np.corrcoef(df['Height'].ffill(),df['Weight'])

दुर्भाग्यवश, हमें कोई परिणाम प्राप्त नहीं हुआ - केवल कुछ अजीब `nan` मान। इसका कारण यह है कि हमारी श्रृंखला के कुछ मान अनिर्धारित हैं, जिन्हें `nan` के रूप में प्रदर्शित किया गया है, जो ऑपरेशन के परिणाम को भी अनिर्धारित बना देता है। मैट्रिक्स को देखकर हम देख सकते हैं कि `Weight` समस्या वाले कॉलम हैं, क्योंकि `Height` मानों के बीच स्व-सहसंबंध की गणना की गई है।

> यह उदाहरण **डेटा तैयारी** और **सफाई** के महत्व को दर्शाता है। उचित डेटा के बिना हम कुछ भी गणना नहीं कर सकते।

आइए `fillna` विधि का उपयोग करके लापता मान भरें, और सहसंबंध की गणना करें: 


In [ ]:
np.corrcoef(df['Height'].ffill(), df['Weight'])

वास्तव में एक सहसंबंध है, लेकिन हमारे कृत्रिम उदाहरण जितना मजबूत नहीं है। वास्तव में, यदि हम एक मान को दूसरे के खिलाफ स्कैटर प्लॉट देखें, तो यह संबंध बहुत कम स्पष्ट होगा:


In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(df['Weight'],df['Height'])
plt.xlabel('Weight')
plt.ylabel('Height')
plt.tight_layout()
plt.show()

## निष्कर्ष

इस नोटबुक में हमने डेटा पर मूलभूत क्रियाएं करने के तरीके सीखे हैं ताकि सांख्यिकीय कार्यों की गणना की जा सके। अब हमें पता है कि कुछ परिकल्पनाओं को साबित करने के लिए गणित और सांख्यिकी के एक मजबूत उपकरण का उपयोग कैसे करें, और डेटा नमूने के आधार पर यादृच्छिक चर के लिए विश्वास अंतराल कैसे गणना करें। 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
